### Visualización de Flujos Logísticos con Python

🎯 El Desafío

El objetivo es explorar cómo las características lineales (carreteras) pueden comunicar información compleja mediante el uso de:

- Grosor (Thickness): Representa el volumen de carga (toneladas de cosecha).

- Color: Identifica el rubro agrícola (Maíz, Soya, Trigo).

- Dirección: Animada mediante AntPath para mostrar el flujo hacia el centro de acopio.

🛠️ Tecnologías Utilizadas
- Python 3.x

- OSMnx: Descarga y manipulación de redes viales de OpenStreetMap.

- NetworkX: Implementación del algoritmo de Dijkstra para el cálculo de la ruta más corta.

- Folium: Visualización geoespacial interactiva en entorno web.

- Leaflet.js: Motor del mapa interactivo (vía Folium).

🚀 Características Principales

- Rutas Reales: A diferencia de las líneas rectas, este modelo calcula trayectorias sobre la red vial real de la región.

- Conectividad de Red: Implementa lógica de teoría de grafos para asegurar que cada punto de origen esté conectado al componente más fuerte de la red vial.

- UI/UX Interactiva: * Mapa base Dark Matter para alto contraste.

- Control de capas para filtrar por tipo de cultivo.

In [11]:
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import AntPath

In [27]:
# 1. Configuración de zona (Región Núcleo Agrícola)
centro_coords = (-33.89, -60.57) 
print("Descargando infraestructura vial de OpenStreetMap...")

# Descarga y limpieza de red (Garantiza que todas las rutas conecten)
G_raw = ox.graph_from_point(centro_coords, dist=20000, network_type='drive')
G_weak = G_raw.to_undirected()
largest_component = max(nx.connected_components(G_weak), key=len)
G = G_raw.subgraph(largest_component).copy()

nodo_planta = ox.nearest_nodes(G, centro_coords[1], centro_coords[0])

Descargando infraestructura vial de OpenStreetMap...


In [26]:
# 2. Datos de las 8 Granjas (Fuera de carretera)
granjas_data = [
    {"nombre": "Finca El Sol", "coords": (-33.80, -60.48), "ton": 80, "rubro": "Maíz", "color": "#f1c40f"},
    {"nombre": "Don Pedro", "coords": (-33.97, -60.42), "ton": 40, "rubro": "Soya", "color": "#2ecc71"},
    {"nombre": "La Posta", "coords": (-33.83, -60.78), "ton": 120, "rubro": "Trigo", "color": "#e67e22"},
    {"nombre": "El Trébol", "coords": (-34.05, -60.58), "ton": 65, "rubro": "Maíz", "color": "#f1c40f"},
    {"nombre": "Santa Rita", "coords": (-33.75, -60.68), "ton": 90, "rubro": "Soya", "color": "#2ecc71"},
    {"nombre": "Campo Largo", "coords": (-33.88, -60.85), "ton": 110, "rubro": "Trigo", "color": "#e67e22"},
    {"nombre": "Las Lilas", "coords": (-34.08, -60.38), "ton": 35, "rubro": "Maíz", "color": "#f1c40f"},
    {"nombre": "El Amanecer", "coords": (-33.72, -60.55), "ton": 55, "rubro": "Soya", "color": "#2ecc71"}
]

In [28]:
# 3. Inicializar Mapa (Dark Mode para resaltar líneas)
m = folium.Map(location=centro_coords, zoom_start=11, tiles="cartodb dark_matter")

# Título Superior
titulo_html = '''
    <div style="position: fixed; top: 15px; left: 50%; transform: translateX(-50%); z-index:9999; 
                background-color: rgba(0,0,0,0.7); color: white; padding: 10px 25px; 
                border-radius: 20px; border: 1px solid #555; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
        <h3 style="margin:0; font-size:18px;"><b>LOGÍSTICA AGROINDUSTRIAL: FLUJO DE COSECHA</b></h3>
    </div>
    '''
m.get_root().html.add_child(folium.Element(titulo_html))

# Pie de Página (Centrado y pequeño)
footer_html = '''
    <div style="position: fixed; bottom: 15px; left: 50%; transform: translateX(-50%); z-index:9999; 
                color: #888; font-size: 9px; font-family: 'Arial'; letter-spacing: 1px;">
        Autor: @alexisdaraujoc | Fuente: OSM | © 2026
    </div>
    '''
m.get_root().html.add_child(folium.Element(footer_html))

# Grupos de Capas
grupos = {
    "Maíz": folium.FeatureGroup(name='<span style="color:#f1c40f">●</span> Maíz').add_to(m),
    "Soya": folium.FeatureGroup(name='<span style="color:#2ecc71">●</span> Soya').add_to(m),
    "Trigo": folium.FeatureGroup(name='<span style="color:#e67e22">●</span> Trigo').add_to(m)
}

In [29]:
# 4. Procesamiento de Rutas y Marcadores
for g in granjas_data:
    try:
        n_granja = ox.nearest_nodes(G, g["coords"][1], g["coords"][0])
        path = nx.shortest_path(G, n_granja, nodo_planta, weight='length')
        
        # Geometría: [Posición Granja] -> [Nodos de Carretera]
        ruta_coords = [list(g["coords"])] + [[G.nodes[n]['y'], G.nodes[n]['x']] for n in path]
        
        # Animación AntPath
        AntPath(
            locations=ruta_coords,
            color=g["color"],
            pulse_color='#ffffff',
            weight=g["ton"]/8,
            delay=1100,
            tooltip=f"<b>{g['nombre']}</b><br>Carga: {g['ton']}t"
        ).add_to(grupos[g["rubro"]])
        
        # Marcador Granja
        folium.Marker(
            location=g["coords"],
            icon=folium.Icon(color='black', icon_color=g["color"], icon='leaf', prefix='fa'),
            popup=g["nombre"]
        ).add_to(grupos[g["rubro"]])
        
    except Exception:
        continue

In [30]:
# 5. Planta Central e Interfaz
folium.Marker(
    location=centro_coords,
    icon=folium.Icon(color='red', icon='industry', prefix='fa'),
    popup="PLANTA CENTRAL"
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# Guardar archivo
m.save("mapa_agro_logistica_final.html")
print("✅ Mapa generado exitosamente como 'mapa_agro_logistica_final.html'")

✅ Mapa generado exitosamente como 'mapa_agro_logistica_final.html'
